In [2]:
import pandas as pd

In [4]:
from pathlib import Path


csv_files = sorted(path for path in Path.cwd().rglob("*.csv") if "data_filtered" not in path.relative_to(Path.cwd()).parts)
dataframes = {}

for csv_path in csv_files:
    relative_path = csv_path.relative_to(Path.cwd())
    print(f"\n--- {relative_path} ---")
    try:
        dataframes[str(relative_path)] = pd.read_csv(csv_path)
        print(dataframes[str(relative_path)].head())
    except Exception as exc:
        print(f"Could not read {relative_path}: {exc}")



--- nfl-big-data-bowl-2027\combine_results.csv ---
   draft_year  nfl_id combine_position  combine_height  combine_weight  \
0        2023   56040               TE          77.875             245   
1        2023   56010               DB          70.625             191   
2        2023   56085               TE          79.375             255   
3        2023   56056               WR          72.250             192   
4        2023   56118               WR          71.625             191   

   hand_size  arm_length  wing_span  ten_yd_split  forty  vertical  \
0     10.000      32.250     79.750          1.60   4.84      38.5   
1      8.750      29.625     72.250          1.58   4.59      33.5   
2     10.250      34.000     83.125          1.57   4.55      40.0   
3      9.625      31.875     76.125          1.51   4.33       NaN   
4      9.500      31.000     75.000          1.52   4.54      38.5   

   broad_jump  three_cone  short_shuttle  bench_reps  ngs_athleticism_score  \
0  

In [5]:
# Run after the CSV-loading cell. Original dataframes are preserved.
# Define the cohort by roster position, not alignment on a particular play.
def normalized_ids(series):
    return pd.to_numeric(series, errors="coerce").astype("Int64")

te_ids = set()
position_sources = [
    (name, df) for name, df in dataframes.items()
    if Path(name).name.lower() == "players.csv"
    and {"nfl_id", "nfl_position"}.issubset(df.columns)
]
# Fall back to Combine position if the roster file was not loaded.
if not position_sources:
    position_sources = [
        (name, df) for name, df in dataframes.items()
        if {"nfl_id", "combine_position"}.issubset(df.columns)
    ]
if not position_sources:
    raise ValueError("Load players.csv or a CSV with nfl_id and combine_position first.")

for name, df in position_sources:
    position_col = "nfl_position" if "nfl_position" in df.columns else "combine_position"
    is_te = df[position_col].astype("string").str.strip().str.upper().isin(["TE", "TIGHT END", "TIGHTEND"])
    te_ids.update(normalized_ids(df.loc[is_te, "nfl_id"]).dropna().tolist())

te_dataframes = {}
for name, df in dataframes.items():
    if "nfl_id" in df.columns:
        te_dataframes[name] = df.loc[normalized_ids(df["nfl_id"]).isin(te_ids)].copy()

# Game-level tables contain context rather than individual players.
te_game_ids = set()
for df in te_dataframes.values():
    if "game_id" in df.columns:
        te_game_ids.update(normalized_ids(df["game_id"]).dropna().tolist())

for name, df in dataframes.items():
    if "nfl_id" not in df.columns:
        if "game_id" in df.columns:
            te_dataframes[name] = df.loc[normalized_ids(df["game_id"]).isin(te_game_ids)].copy()
        else:
            print(f"Skipped {name}: no player or game ID for filtering.")

print(f"Tight end cohort: {len(te_ids)} players")
for name, df in te_dataframes.items():
    print(f"\n--- {name}: {len(df):,} rows ---")
    print(df.head())


Tight end cohort: 42 players

--- nfl-big-data-bowl-2027\combine_results.csv: 42 rows ---
    draft_year  nfl_id combine_position  combine_height  combine_weight  \
0         2023   56040               TE          77.875             245   
2         2023   56085               TE          79.375             255   
6         2023   55900               TE          76.500             249   
24        2023   56363               TE          76.625             250   
28        2023   56012               TE          78.500             248   

    hand_size  arm_length  wing_span  ten_yd_split  forty  vertical  \
0      10.000      32.250     79.750          1.60   4.84      38.5   
2      10.250      34.000     83.125          1.57   4.55      40.0   
6       9.500      31.625     76.250          1.66   4.70      32.5   
24     10.125      32.750     80.500           NaN    NaN       NaN   
28      9.500      31.500     77.375          1.61   4.69      33.5   

    broad_jump  three_cone  shor

In [6]:
from pathlib import Path

# Run after the tight end filtering cell.
output_dir = Path.cwd() / "data_filtered"
output_dir.mkdir(parents=True, exist_ok=True)

# Preserve relative paths so identically named source files do not collide.
for name, df in te_dataframes.items():
    output_path = output_dir / Path(name)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)
    print(f"Saved {len(df):,} rows to {output_path.relative_to(Path.cwd())}")


Saved 42 rows to data_filtered\nfl-big-data-bowl-2027\combine_results.csv
Saved 44,498 rows to data_filtered\nfl-big-data-bowl-2027\combine_tracking.csv
Saved 117,744 rows to data_filtered\nfl-big-data-bowl-2027\game_tracking_2023.csv
Saved 282,234 rows to data_filtered\nfl-big-data-bowl-2027\game_tracking_2024.csv
Saved 424,767 rows to data_filtered\nfl-big-data-bowl-2027\game_tracking_2025.csv
Saved 42 rows to data_filtered\nfl-big-data-bowl-2027\player_career_successes.csv
Saved 10,410 rows to data_filtered\nfl-big-data-bowl-2027\player_play.csv
Saved 42 rows to data_filtered\nfl-big-data-bowl-2027\players.csv
Saved 644 rows to data_filtered\nfl-big-data-bowl-2027\games.csv
